A notebook for prompting of SOTA LLMs in zero and few shot setup for NER.

## Data Loading

In [7]:
!pip install scikit-multilearn

  Using cached scikit_multilearn-0.2.0-py3-none-any.whl.metadata (6.0 kB)
Using cached scikit_multilearn-0.2.0-py3-none-any.whl (89 kB)


In [8]:

import json
import re

from tqdm import tqdm

import os
from pathlib import Path

from dataset_processing import process_llm_jsonl_results, shorten_name

import time

with open("not_to_upload.txt", "r") as f:
    API_KEY = f.read().split("\n")
    
with open("system_prompt.md", "r") as f:
    system_prompt = f.read()
    
with open("definitions.txt", "r") as f:
    definitions_str = f.read()

definitions_dict = [{"label": definition.split(" --- ")[0], "definition":definition.split(" --- ")[1]} for definition in definitions_str.split("\n")]

def create_string(def_dict):
    definitions_string = ""
    for ent in def_dict:
        definitions_string = definitions_string + ent["label"] + " - " + ent["definition"] + "\n"
        
    return definitions_string

system_prompt_withdefinitions = re.sub(r"\[TAXONOMY - DEFINITIONS - RULES\]", create_string(definitions_dict), system_prompt)

In [9]:
print(system_prompt_withdefinitions)

### ROLE
You are an expert Named Entity Recognition (NER) system. Your task is to extract entities from the user's input text and classify them according to the provided taxonomy.

### TAXONOMY & RULES
Asset - is an object or service of value to humans that can get destroyed or diminished by climate disasters/hazards. Key categories are health, buildings, infrastructure, and crops or livestock.
Body of Water - is a distinct volume or mass of water, whether naturally occurring (like an ocean or river) or contained within a man-made system (like a reservoir or effluent flow). This includes specific named bodies, general types, and scientifically defined water masses.
Body Part - is a structural component of a living organism (plant, animal, or human), which is not the whole organism but a distinct anatomical or morphological part.
Chemical - is a substance with a distinct molecular composition, including elements, compounds, ions, biological molecules, and mixtures. It also encompasses c

In [10]:
DATA_DIR = "RESULTS/annots_v2"

In [11]:
files = os.listdir(DATA_DIR)

sentences = {}

for file in files:
    file_dir = Path.joinpath(Path(DATA_DIR),Path(file))
    print(file_dir)

    with open(file_dir, "r") as f:
        data_json = json.load(f)
        
    for i in range(len(data_json)):
        paper_id = data_json[i]["data"]["paper_id"]
        sentence_id = data_json[i]["data"]["sentence_id"]
        
        compound = f"{paper_id}-{sentence_id}"
        # print(compound)
        sentences[f"{compound}"]= data_json[i]["data"]["sentence"]

print(len(sentences))



RESULTS\annots_v2\mmp_50_Ass_Pol.json
RESULTS\annots_v2\mmp_50_BodPar_Che_Dis_Org_Eco.json
RESULTS\annots_v2\mmp_50_EneSou_MetPhe_NatDis_NatPhe.json
RESULTS\annots_v2\mmp_50_Loc_GeoFea_BodOfWat_TimPer_Sat.json
RESULTS\annots_v2\mmp_50_MatExp_MeaDev_PhyPhe_Qua.json
RESULTS\annots_v2\mmp_50_Met_FieOfStu_IntArt.json
RESULTS\annots_v2\mmp_50_PhyArt_Org_Per_Sys.json
192


### Helper Functions

In [12]:
MODEL_NAME = "Unnamed"
def query_llm_gemini(client, input_sentence, system_prompt, model_name=MODEL_NAME):
    """
    Sends the sentence to Gemini and returns a dictionary containing 
    the input and the raw response text.
    """
    try:
        response = client.models.generate_content(
            model=model_name,
            contents=input_sentence,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                response_mime_type="application/json", 
                temperature=0.1
            )
        )
        
        # Return a structured package
        return {
            "input_text": input_sentence,
            "raw_response": response.text,
            "success": True
        }
        
    except Exception as e:
        print(f"Error processing sentence: {input_sentence[:30]}... | Error: {e}")
        return {
            "input_text": input_sentence,
            "raw_response": None,
            "success": False,
            "error": str(e)
        }

def query_llm_openai(client_openai, input_sentence, system_prompt, model_name=MODEL_NAME):
    """
    Sends the sentence to OpenAI and returns a structured package 
    compatible with the existing parse_and_align_spans function.
    """
    try:
        response = client_openai.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": input_sentence}
            ],
            # NOTE: If your system prompt asks for a LIST [...], remove response_format.
            # If it asks for an OBJECT {"entities": [...]}, keep it.
            # OpenAI 'json_object' mode strictly requires the output to be a dictionary {}.
            response_format={"type": "json_object"}, 
            temperature=0.1
        )
        
        raw_content = response.choices[0].message.content
        
        return {
            "input_text": input_sentence,
            "raw_response": raw_content,
            "success": True
        }
        
    except Exception as e:
        print(f"Error processing sentence: {input_sentence[:30]}... | Error: {e}")
        return {
            "input_text": input_sentence,
            "raw_response": None,
            "success": False,
            "error": str(e)
        }
    
def query_llm_openai_freshmodels(client_openai, input_sentence, system_prompt, model_name=MODEL_NAME):
    """
    Sends the sentence to OpenAI (Responses API) and returns a structured package
    compatible with the existing parse_and_align_spans function.
    """
    try:
        response = client_openai.responses.create(
            model=model_name,
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": input_sentence}
            ],
            # Same semantics as before: enforce JSON object output
            # response_format={"type": "json_object"},
            # temperature=0.1
        )

        # Responses API does NOT use choices[0].message.content
        # output_text is a convenience accessor that concatenates text blocks
        raw_content = response.output_text

        return {
            "input_text": input_sentence,
            "raw_response": raw_content,
            "success": True
        }

    except Exception as e:
        print(f"Error processing sentence: {input_sentence[:30]}... | Error: {e}")
        return {
            "input_text": input_sentence,
            "raw_response": None,
            "success": False,
            "error": str(e)
        }
        
def query_llm_deepseek(client, input_sentence, system_prompt, model_name="deepseek-chat"):
    """
    Sends the sentence to DeepSeek and returns a dictionary containing 
    the input and the raw response text.
    """
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": input_sentence}
            ],
            response_format={"type": "json_object"}, # Forces valid JSON output
            temperature=0.1,
            stream=False
        )
        
        # Extract the content string
        response_text = response.choices[0].message.content
        
        # Return a structured package
        return {
            "input_text": input_sentence,
            "raw_response": response_text,
            "success": True
        }
        
    except Exception as e:
        print(f"Error processing sentence: {input_sentence[:30]}... | Error: {e}")
        return {
            "input_text": input_sentence,
            "raw_response": None,
            "success": False,
            "error": str(e)
        }
        
def query_llm_claude(client_anthropic, input_sentence, system_prompt, model_name="claude-sonnet-4-20250514"):
    """
    Sends the sentence to Claude and returns a structured package 
    compatible with the existing parse_and_align_spans function.
    """
    try:
        response = client_anthropic.messages.create(
            model=model_name,
            max_tokens=1000,
            system=system_prompt,
            messages=[
                {"role": "user", "content": input_sentence}
            ],
            temperature=0.1
        )
        # Extract text content from the response
        raw_content = response.content[0].text
        
        return {
            "input_text": input_sentence,
            "raw_response": raw_content,
            "success": True
        }
    except Exception as e:
        print(f"Error processing sentence: {input_sentence[:30]}... | Error: {e}")
        return {
            "input_text": input_sentence,
            "raw_response": None,
            "success": False,
            "error": str(e)
        }
 

def parse_and_align_spans(result_package):
    """
    Parses the JSON response (List or Dict) and calculates exact start/end indices.
    """
    if not result_package.get("success") or not result_package.get("raw_response"):
        return []

    input_text = result_package["input_text"]
    raw_json = result_package["raw_response"]
    
    extracted_entities = []

    try:
        # 1. Clean and Parse JSON
        clean_json = raw_json.replace("```json", "").replace("```", "").strip()
        data = json.loads(clean_json)

        # --- FIX FOR OPENAI vs GEMINI STRUCTURE ---
        entity_list = []
        
        if isinstance(data, list):
            # Gemini usually returns [ {item}, {item} ]
            entity_list = data
        elif isinstance(data, dict):
            # OpenAI 'json_object' mode enforces { "key": [items] }
            # We look for common keys or just take the first list we find.
            if "entities" in data:
                entity_list = data["entities"]
            elif "items" in data:
                entity_list = data["items"]
            elif "results" in data:
                entity_list = data["results"]
            else:
                # Fallback: Find the first value that is a list
                for key, value in data.items():
                    if isinstance(value, list):
                        entity_list = value
                        break
        # -------------------------------------------

        # 2. Iterative Span Finding
        current_cursor = 0
        
        for item in entity_list:
            # Safety check: ensure item is actually a dict
            if not isinstance(item, dict):
                continue

            entity_text = item.get('entity_text', '').strip()
            category = item.get('category', 'Unknown')
            reasoning = item.get('reasoning', '')

            if not entity_text:
                continue

            # Escape special regex characters (e.g., "+", "(", ")")
            pattern = re.escape(entity_text)
            
            # Search ONLY after the current cursor to handle duplicates in order
            match = re.search(pattern, input_text[current_cursor:])
            
            if match:
                relative_start = match.start()
                relative_end = match.end()
                
                abs_start = current_cursor + relative_start
                abs_end = current_cursor + relative_end
                
                extracted_entities.append({
                    "entity_text": entity_text,
                    "category": category,
                    "reasoning": reasoning,
                    "span": (abs_start, abs_end)
                })
                
                # Advance cursor
                current_cursor = abs_end
            else:
                # Fallback: Search from beginning if not found linearly
                # (Useful if LLM messed up the order)
                print(f"Warning: '{entity_text}' not found strictly after index {current_cursor}. Trying full search.")
                fallback_match = re.search(pattern, input_text)
                if fallback_match:
                     extracted_entities.append({
                        "entity_text": entity_text,
                        "category": category,
                        "reasoning": reasoning,
                        "span": (fallback_match.start(), fallback_match.end()),
                        "note": "Out of order match"
                    })

    except json.JSONDecodeError:
        print(f"JSON Parsing failed for input: {input_text[:20]}...")
    except Exception as e:
        print(f"Error aligning spans: {e}")
    
    return extracted_entities
           



## Gemini

In [18]:
# !pip install google-genai
from google import genai
from google.genai import types

In [19]:
client = genai.Client(api_key=API_KEY[0])
SYSTEM_INSTRUCTION = system_prompt_withdefinitions
MODEL_NAME = "gemma-4-31b-it"# , "gemini-3-pro-preview" # "gemini-2.5-pro" # "gemini-3-pro-preview" # 

RPM = 25
RPD = 250  
OUTPUT_FILE = f"ner_results_{shorten_name(MODEL_NAME)}.jsonl"

sleeptime = (60 / RPM) + 1 
requests_count = 0


In [20]:
model_info = client.models.get(model=MODEL_NAME)

print(f"--- Model Information: {MODEL_NAME} ---")
print(f"Name: {model_info.name}")
print(f"Display Name: {model_info.display_name}")
print(f"Version: {model_info.version}")
print(f"Description: {model_info.description}")
print(f"Input Token Limit: {model_info.input_token_limit}")
print(f"Output Token Limit: {model_info.output_token_limit}")
print("-" * 30)

print("--- Searching for available Model Snapshots ---")

--- Model Information: gemma-4-31b-it ---
Name: models/gemma-4-31b-it
Display Name: Gemma 4 31B IT
Version: 001
Description: Gemma 4 31B IT
Input Token Limit: 262144
Output Token Limit: 32768
------------------------------
--- Searching for available Model Snapshots ---


In [ ]:
processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    print(f"Scanning {OUTPUT_FILE} for existing progress...")
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f_in:
        for line in f_in:
            try:
                data = json.loads(line)
                # Check if ID exists and raw_llm_response is valid (not None)
                if data.get("compound_id") and data.get("raw_llm_response") is not None:
                    processed_ids.add(data["compound_id"])
            except json.JSONDecodeError:
                continue # Skip corrupted lines
print(f"Found {len(processed_ids)} already processed sentences.")

sentences_to_process = list(sentences.keys()) # specific list of keys to control order if needed

remaining_work = len([sid for sid in sentences_to_process if sid not in processed_ids])
to_do_count = min(remaining_work, RPD - requests_count) # assumes requests_count starts at 0
print(f"Starting job. Actually processing: {to_do_count}. Est. time: {to_do_count * sleeptime / 60:.1f} minutes.")

# Open file in APPEND mode so we don't overwrite if we restart the script
with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:
    
    for i, sentence_id in tqdm(enumerate(sentences_to_process)):
        
        # 1. Check Hard Limit        
        if requests_count >= RPD:
            print("Daily/Batch limit reached. Stopping.")
            break
        
        # Skip already processed items
        if sentence_id in processed_ids:
            continue

        sentence = sentences[sentence_id]
        print(f"[{i+1}/{len(sentences)}] Processing ID {sentence_id}...")

        # 2. Step A: Query
        # (Assuming query_llm_gemini is defined as in the previous turn)
        raw_result = query_llm_gemini(client, sentence, SYSTEM_INSTRUCTION, MODEL_NAME)
        
        # 3. Step B: Parse
        # Only parse if the query was successful
        if raw_result['success']:
            final_entities = parse_and_align_spans(raw_result)
        else:
            final_entities = [] # Or handle error specifically

        # 4. Step C: Prepare Entry
        entry = {
            "compound_id": sentence_id,
            "input_text": sentence,
            "raw_llm_response": raw_result.get('raw_response'), 
            "processed_entities": final_entities,
            "timestamp": time.time()
        }

        # 5. Write to File IMMEDIATELY (JSONL format)
        f_out.write(json.dumps(entry) + "\n")
        f_out.flush()

        # 6. Rate Limiting
        requests_count += 1
        time.sleep(sleeptime)

print(f"\nDone. Results saved to {OUTPUT_FILE}")

Found 0 already processed sentences.
Starting job. Actually processing: 192. Est. time: 10.9 minutes.


0it [00:00, ?it/s]

[1/192] Processing ID 36618-21...


1it [01:43, 103.03s/it]

[2/192] Processing ID 2416-11...


...

191it [5:19:26, 41.28s/it]

[192/192] Processing ID 33310-38...


192it [5:19:57, 99.99s/it]


Done. Results saved to ner_results_gemma_4_31b_it.jsonl


: 

## ChatGPT

In [17]:
# !pip install openai
from openai import OpenAI

In [20]:
# 1. Setup
client = OpenAI(api_key=API_KEY[1]) # Using index 1 as requested

SYSTEM_INSTRUCTION = system_prompt_withdefinitions
MODEL_NAME = "gpt-5.1" # "gpt-5.2-pro" # "gpt-5.1" # 

RPM = 500
RPD = 10000
OUTPUT_FILE = f"ner_results_{shorten_name(MODEL_NAME)}.jsonl"

sleeptime = (60 / RPM) + 1 
requests_count = 0

In [21]:
import datetime

print(f"--- Model Information: {MODEL_NAME} ---")
try:
    # 1. Retrieve specific model info
    # Note: OpenAI API does NOT return context limits or descriptions programmatically.
    # You must consult documentation for those values.
    model_info = client.models.retrieve(MODEL_NAME)
    
    print(f"ID (Actual Name): {model_info.id}")
    print(f"Owned By: {model_info.owned_by}")
    
    # Convert Unix timestamp to readable date
    created_date = datetime.datetime.fromtimestamp(model_info.created).strftime('%Y-%m-%d %H:%M:%S')
    print(f"Created At: {created_date}")
    print("-" * 30)

except Exception as e:
    print(f"Could not retrieve metadata for {MODEL_NAME}. Error: {e}")


print("--- Searching for available Model Snapshots ---")
# This is crucial for scientific reproducibility. 
# Look for IDs with dates (e.g., -0613, -2024-08-06) in the output below.
try:
    # Fetch all models
    all_models = client.models.list()
    
    # Sort them alphabetically for easier reading
    sorted_models = sorted(all_models.data, key=lambda x: x.id)
    
    for m in sorted_models:
        # Filter to show only relevant models (e.g., matching your base model name)
        # We split by "-" to match "gpt-5" against "gpt-5.2-pro" etc.
        base_search = MODEL_NAME.split("-")[0] 
        
        if base_search in m.id:
            created_dt = datetime.datetime.fromtimestamp(m.created).strftime('%Y-%m-%d')
            print(f"ID: {m.id:<30} | Created: {created_dt}")

except Exception as e:
    print(f"Error listing models: {e}")

--- Model Information: gpt-5.1 ---
ID (Actual Name): gpt-5.1
Owned By: system
Created At: 2025-11-10 19:51:13
------------------------------
--- Searching for available Model Snapshots ---
ID: chatgpt-4o-latest              | Created: 2024-08-13
ID: chatgpt-image-latest           | Created: 2025-12-16
ID: gpt-3.5-turbo                  | Created: 2023-02-28
ID: gpt-3.5-turbo-0125             | Created: 2024-01-23
ID: gpt-3.5-turbo-1106             | Created: 2023-11-02
ID: gpt-3.5-turbo-16k              | Created: 2023-05-11
ID: gpt-3.5-turbo-instruct         | Created: 2023-08-24
ID: gpt-3.5-turbo-instruct-0914    | Created: 2023-09-07
ID: gpt-4                          | Created: 2023-06-27
ID: gpt-4-0125-preview             | Created: 2024-01-23
ID: gpt-4-0613                     | Created: 2023-06-12
ID: gpt-4-1106-preview             | Created: 2023-11-02
ID: gpt-4-turbo                    | Created: 2024-04-06
ID: gpt-4-turbo-2024-04-09         | Created: 2024-04-08
ID: gpt-4-tur

In [ ]:
processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    print(f"Scanning {OUTPUT_FILE} for existing progress...")
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f_in:
        for line in f_in:
            try:
                data = json.loads(line)
                # Check if ID exists and raw_llm_response is valid (not None)
                if data.get("compound_id") and data.get("raw_llm_response") is not None:
                    processed_ids.add(data["compound_id"])
            except json.JSONDecodeError:
                continue # Skip corrupted lines
print(f"Found {len(processed_ids)} already processed sentences.")

sentences_to_process = list(sentences.keys()) 

remaining_work = len([sid for sid in sentences_to_process if sid not in processed_ids])
to_do_count = min(remaining_work, RPD - requests_count) # assumes requests_count starts at 0
print(f"Starting job. Actually processing: {to_do_count}. Est. time: {to_do_count * sleeptime / 60:.1f} minutes.")

# --- MAIN EXECUTION LOOP ---

# Open file in APPEND mode
with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:
    
    for i, sentence_id in enumerate(sentences_to_process):
        
        # 1. Check Hard Limit
        if requests_count >= RPD:
            print("Daily/Batch limit reached. Stopping.")
            break
        
        # Skip already processed items 
        if sentence_id in processed_ids:
            continue

        sentence = sentences[sentence_id]
        print(f"[{i+1}/{len(sentences)}] Processing ID {sentence_id}...")
        # 2. Step A: Query
        raw_result = query_llm_openai_freshmodels(client, sentence, SYSTEM_INSTRUCTION, MODEL_NAME)
        
        # 3. Step B: Parse
        # Using the same parser function from the previous setup
        # print(raw_result)
        if raw_result['success']:
            # Note: If OpenAI returns a wrapper dict (e.g. {"items": [...]}) due to json_object mode,
            # ensure parse_and_align_spans handles dictionary inputs, or use raw_result['raw_response'] logic.
            final_entities = parse_and_align_spans(raw_result)
        else:
            final_entities = []

        # 4. Step C: Prepare Entry
        entry = {
            "compound_id": sentence_id,
            "input_text": sentence,
            "raw_llm_response": raw_result.get('raw_response'), 
            "processed_entities": final_entities,
            "timestamp": time.time()
        }

        # 5. Write to File IMMEDIATELY
        f_out.write(json.dumps(entry) + "\n")
        f_out.flush()

        # 6. Rate Limiting
        requests_count += 1
        time.sleep(sleeptime)

print(f"\nDone. Results saved to {OUTPUT_FILE}")

## DeepSeek

In [23]:
from openai import OpenAI

In [24]:
client = OpenAI(
    api_key=API_KEY[2], 
    base_url="https://api.deepseek.com"
)

SYSTEM_INSTRUCTION = system_prompt_withdefinitions
MODEL_NAME = "deepseek-reasoner" # or "deepseek-coder"

# Rate Limiting & File Config
RPM = 100  # Adjust based on your DeepSeek tier limits
RPD = 250 
OUTPUT_FILE = f"ner_results_{shorten_name(MODEL_NAME)}.jsonl"

sleeptime = (60 / RPM) + 1 
requests_count = 0

In [ ]:
processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    print(f"Scanning {OUTPUT_FILE} for existing progress...")
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f_in:
        for line in f_in:
            try:
                data = json.loads(line)
                # Check if ID exists and raw_llm_response is valid (not None)
                if data.get("compound_id") and data.get("raw_llm_response") is not None:
                    processed_ids.add(data["compound_id"])
            except json.JSONDecodeError:
                continue # Skip corrupted lines
print(f"Found {len(processed_ids)} already processed sentences.")

# Prepare the work queue
sentences_to_process = list(sentences.keys()) 

# Calculate remaining work
remaining_work = len([sid for sid in sentences_to_process if sid not in processed_ids])
to_do_count = min(remaining_work, RPD - requests_count) 
print(f"Starting job. Actually processing: {to_do_count}. Est. time: {to_do_count * sleeptime / 60:.1f} minutes.")

# ==========================================
# 3. Main Processing Loop
# ==========================================
# Open file in APPEND mode so we don't overwrite if we restart the script
with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:
    
    for i, sentence_id in tqdm(enumerate(sentences_to_process)):
        
        # 1. Check Hard Limit        
        if requests_count >= RPD:
            print("Daily/Batch limit reached. Stopping.")
            break
        
        # Skip already processed items
        if sentence_id in processed_ids:
            continue

        sentence = sentences[sentence_id]
        # Optional: Print less verbose status if tqdm is running
        # print(f"[{i+1}/{len(sentences)}] Processing ID {sentence_id}...")

        # 2. Step A: Query (Switched to DeepSeek function)
        # Using the wrapper function defined in the previous step
        raw_result = query_llm_deepseek(client, sentence, SYSTEM_INSTRUCTION, MODEL_NAME)
        
        # 3. Step B: Parse
        # Only parse if the query was successful
        if raw_result['success']:
            # Assumes parse_and_align_spans works with the JSON string returned by DeepSeek
            final_entities = parse_and_align_spans(raw_result)
        else:
            final_entities = [] # Or handle error specifically

        # 4. Step C: Prepare Entry
        entry = {
            "compound_id": sentence_id,
            "input_text": sentence,
            "raw_llm_response": raw_result.get('raw_response'), 
            "processed_entities": final_entities,
            "timestamp": time.time(),
            "model": MODEL_NAME
        }

        # 5. Write to File IMMEDIATELY (JSONL format)
        f_out.write(json.dumps(entry) + "\n")
        f_out.flush()

        # 6. Rate Limiting
        requests_count += 1
        time.sleep(sleeptime)

print(f"\nDone. Results saved to {OUTPUT_FILE}")

## Claude

In [2]:
# !pip install anthropic
from anthropic import Anthropic

In [ ]:
client = Anthropic(api_key=API_KEY[3])  

SYSTEM_INSTRUCTION = system_prompt_withdefinitions
MODEL_NAME = "claude-sonnet-4-20250514"  # or "claude-opus-4-20250514"

RPM = 50  # Claude tier 1: 50 requests per minute
RPD = 1000  # Claude tier 1: 1000 requests per day (adjust based on your tier)

OUTPUT_FILE = f"ner_results_{shorten_name(MODEL_NAME)}.jsonl"

sleeptime = (60 / RPM) + 1 
requests_count = 0

In [ ]:
processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    print(f"Scanning {OUTPUT_FILE} for existing progress...")
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f_in:
        for line in f_in:
            try:
                data = json.loads(line)
                # Check if ID exists and raw_llm_response is valid (not None)
                if data.get("compound_id") and data.get("raw_llm_response") is not None:
                    processed_ids.add(data["compound_id"])
            except json.JSONDecodeError:
                continue  # Skip corrupted lines
    print(f"Found {len(processed_ids)} already processed sentences.")

sentences_to_process = list(sentences.keys()) 
remaining_work = len([sid for sid in sentences_to_process if sid not in processed_ids])
to_do_count = min(remaining_work, RPD - requests_count)  # assumes requests_count starts at 0
print(f"Starting job. Actually processing: {to_do_count}. Est. time: {to_do_count * sleeptime / 60:.1f} minutes.")

# --- MAIN EXECUTION LOOP ---
# Open file in APPEND mode
with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:
    for i, sentence_id in enumerate(sentences_to_process):
        # 1. Check Hard Limit
        if requests_count >= RPD:
            print("Daily/Batch limit reached. Stopping.")
            break
        
        # Skip already processed items 
        if sentence_id in processed_ids:
            continue
        
        sentence = sentences[sentence_id]
        print(f"[{i+1}/{len(sentences)}] Processing ID {sentence_id}...")
        
        # 2. Step A: Query - CHANGED TO CLAUDE FUNCTION
        raw_result = query_llm_claude(client, sentence, SYSTEM_INSTRUCTION, MODEL_NAME)
        
        # 3. Step B: Parse
        # Using the same parser function from the previous setup
        if raw_result['success']:
            final_entities = parse_and_align_spans(raw_result)
        else:
            final_entities = []
        
        # 4. Step C: Prepare Entry
        entry = {
            "compound_id": sentence_id,
            "input_text": sentence,
            "raw_llm_response": raw_result.get('raw_response'), 
            "processed_entities": final_entities,
            "timestamp": time.time()
        }
        
        # 5. Write to File IMMEDIATELY
        f_out.write(json.dumps(entry) + "\n")
        f_out.flush()
        
        # 6. Rate Limiting
        requests_count += 1
        time.sleep(sleeptime)

print(f"\nDone. Results saved to {OUTPUT_FILE}")